# Sync Pipeline Outputs → Sandbox bi_data

Copies pipeline dashboard CSVs into `Integrated_Client_Delivery_Sandbox/bi_data/`  
so Tableau, Power BI, and the Excel hub all consume the same refreshed data.

**Run after:** `python run_all_indications.py`  
**Set `DRY_RUN = False`** in Cell 2 to actually write files (default is preview mode).

In [ ]:
# Cell 1 — Imports
import shutil
from datetime import datetime
from pathlib import Path
import pandas as pd
print('Imports OK')

In [ ]:
# Cell 2 — Configuration
DRY_RUN = True   # Set to False to actually write files

PIPELINE_ROOT = Path().resolve().parent  # one level up from scripts/
DASHBOARD_DIR = PIPELINE_ROOT / "output" / "dashboard"
SANDBOX_BI    = (
    PIPELINE_ROOT.parent
    / "Client presentation"
    / "Integrated_Client_Delivery_Sandbox"
    / "bi_data"
)

# Source filename → destination filename
COPY_MAP = {
    "evidence.csv":         "evidence_data.csv",
    "forecast.csv":         "forecast_data.csv",
    "kpi.csv":              "kpi_summary.csv",
    "insights_summary.csv": "metric_summary.csv",
    "source_log.csv":       "evidence_rollup.csv",
    "scenario_registry.csv": "scenario_summary.csv",
}

print(f'Dashboard dir: {DASHBOARD_DIR}')
print(f'Sandbox bi_data: {SANDBOX_BI}')
print(f'Dry run: {DRY_RUN}')

In [ ]:
# Cell 3 — Indication name mapping helper
NAME_MAP = {
    "CLL":                  "Chronic Lymphocytic Leukemia (CLL)",
    "Hodgkin Lymphoma":     "Hodgkin Lymphoma (HL)",
    "Non-Hodgkin Lymphoma": "Non-Hodgkin Lymphoma (excl. DLBCL)",
    "Gastric":              "Gastric Cancer (Stomach Cancer)",
    "Ovarian":              "Ovarian Cancer",
    "Prostate":             "Prostate Cancer",
}

def add_full_name(df: pd.DataFrame) -> pd.DataFrame:
    if "indication" in df.columns and "full_name" not in df.columns:
        df = df.copy()
        df.insert(1, "full_name", df["indication"].map(NAME_MAP).fillna(df["indication"]))
    return df

print('Helpers defined')

In [ ]:
# Cell 4 — Validation check
if not DASHBOARD_DIR.exists():
    print(f'ERROR: Dashboard dir not found: {DASHBOARD_DIR}')
    print('Run the pipeline first: python run_all_indications.py')
elif not SANDBOX_BI.exists():
    print(f'ERROR: Sandbox bi_data not found: {SANDBOX_BI}')
else:
    print('Both source and destination directories exist ✓')
    existing = [f.name for f in DASHBOARD_DIR.glob('*.csv')]
    print(f'Files in dashboard dir: {existing}')

In [ ]:
# Cell 5 — Sync
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
synced, skipped = [], []

for src_name, dst_name in COPY_MAP.items():
    src = DASHBOARD_DIR / src_name
    dst = SANDBOX_BI / dst_name

    if not src.exists():
        skipped.append(src_name)
        print(f'  SKIP  {src_name} (not found)')
        continue

    if DRY_RUN:
        df = pd.read_csv(src)
        print(f'  [dry-run] Would copy {src_name} → {dst_name}  ({len(df):,} rows)')
        synced.append(dst_name)
        continue

    try:
        df = pd.read_csv(src)
        df = add_full_name(df)
        df.to_csv(dst, index=False)
        synced.append(dst_name)
        print(f'  Synced  {src_name} → {dst_name}  ({len(df):,} rows)')
    except Exception as e:
        print(f'  ERROR syncing {src_name}: {e}')
        skipped.append(src_name)

# Update manifest timestamp
manifest = SANDBOX_BI / "dashboard_data_manifest.csv"
if manifest.exists() and not DRY_RUN:
    try:
        df_m = pd.read_csv(manifest)
        if "last_updated" in df_m.columns:
            df_m["last_updated"] = timestamp
        df_m.to_csv(manifest, index=False)
        print(f'  Updated manifest timestamp → {timestamp}')
    except Exception as e:
        print(f'  Warning: manifest update failed: {e}')

prefix = '[dry-run] ' if DRY_RUN else ''
print(f'\n{prefix}Sync complete: {len(synced)} files synced' +
      (f', {len(skipped)} skipped ({", ".join(skipped)})' if skipped else ''))
print(f'Target: {SANDBOX_BI}')